In [ ]:
# ============================================
# DCGAN - 100 EPOCHS (FIXED)
# Run this first
# ============================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import torchvision.transforms as transforms
from PIL import Image

# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Add path
import sys
sys.path.insert(0, '/content/drive/MyDrive/CSE720')

# Configuration
class Config:
    img_size = 64
    batch_size = 8
    dataset_path = '/content/drive/MyDrive/CSE720/EyeGAN'
    num_epochs = 100
    class_names = {
        0: 'Diabetic_Retinopathy',
        1: 'Glaucoma',
        2: 'Healthy',
        3: 'Macular_Scar',
        4: 'Myopia'
    }

cfg = Config()

# Dataset Class
class MedicalDataset(Dataset):
    def __init__(self, root_dir, image_size=64, mode='train', train_ratio=0.8, val_ratio=0.1):
        self.root_dir = root_dir
        self.image_size = image_size
        self.mode = mode

        self.domains = sorted([d.strip() for d in os.listdir(root_dir)
                             if os.path.isdir(os.path.join(root_dir, d))])

        self.image_paths = []
        self.labels = []
        self.domain_to_label = {domain: idx for idx, domain in enumerate(self.domains)}

        for domain in self.domains:
            domain_path = os.path.join(root_dir, domain)
            if not os.path.exists(domain_path):
                continue
            domain_images = [os.path.join(domain_path, img) for img in os.listdir(domain_path)
                           if img.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]

            total_images = len(domain_images)
            train_split = int(total_images * train_ratio)
            val_split = int(total_images * (train_ratio + val_ratio))

            if mode == 'train':
                selected_images = domain_images[:train_split]
            elif mode == 'val':
                selected_images = domain_images[train_split:val_split]
            else:
                selected_images = domain_images[val_split:]

            label = self.domain_to_label[domain]
            self.image_paths.extend(selected_images)
            self.labels.extend([label] * len(selected_images))

        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

        print(f"{mode} mode: {len(self.image_paths)} images")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            image = self.transform(image)
            label = self.labels[idx]
            return image, label, img_path
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            placeholder = torch.zeros(3, self.image_size, self.image_size)
            return placeholder, self.labels[idx], "error_path"

# Load dataset
print("\n" + "="*60)
print("LOADING DATASET")
print("="*60)

train_dataset = MedicalDataset(cfg.dataset_path, cfg.img_size, 'train')
test_dataset = MedicalDataset(cfg.dataset_path, cfg.img_size, 'test')

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=0, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=0, drop_last=True)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# ============================================
# DCGAN MODELS
# ============================================

class Generator(nn.Module):
    def __init__(self, nz=100, ngf=64, nc=3):
        super(Generator, self).__init__()
        self.nz = nz
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. (ngf*8) x 4 x 4
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. (ngf*4) x 8 x 8
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. (ngf*2) x 16 x 16
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. (ngf) x 32 x 32
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
            # state size. (nc) x 64 x 64
        )

    def forward(self, z):
        return self.main(z.view(-1, self.nz, 1, 1))


class Discriminator(nn.Module):
    def __init__(self, ndf=64, nc=3):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            # input is (nc) x 64 x 64
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf) x 32 x 32
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*2) x 16 x 16
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*4) x 8 x 8
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*8) x 4 x 4
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.main(x).view(-1, 1).squeeze(1)


def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


# ============================================
# TRAINING
# ============================================

save_dir = '/content/drive/MyDrive/CSE720/dcgan_results'
os.makedirs(save_dir, exist_ok=True)

nz = 100
netG = Generator(nz=nz).to(device)
netD = Discriminator().to(device)

netG.apply(weights_init)
netD.apply(weights_init)

print(f"\nGenerator parameters: {sum(p.numel() for p in netG.parameters()):,}")
print(f"Discriminator parameters: {sum(p.numel() for p in netD.parameters()):,}")

criterion = nn.BCELoss()
optimizerG = optim.Adam(netG.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizerD = optim.Adam(netD.parameters(), lr=0.0002, betas=(0.5, 0.999))

fixed_noise = torch.randn(64, nz, device=device)

print("\n" + "="*60)
print("TRAINING DCGAN - 100 EPOCHS")
print("="*60)

history = {'g_loss': [], 'd_loss': []}

for epoch in range(100):
    netG.train()
    netD.train()

    epoch_g_loss = 0
    epoch_d_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/100]")

    for i, (data, _, _) in enumerate(loop):
        batch_size = data.size(0)
        real_data = data.to(device)

        real_labels = torch.ones(batch_size, device=device)
        fake_labels = torch.zeros(batch_size, device=device)

        # Train Discriminator with real images
        netD.zero_grad()
        output = netD(real_data)
        errD_real = criterion(output, real_labels)

        # Train Discriminator with fake images
        noise = torch.randn(batch_size, nz, device=device)
        fake = netG(noise)
        output = netD(fake.detach())
        errD_fake = criterion(output, fake_labels)

        errD = errD_real + errD_fake
        errD.backward()
        optimizerD.step()

        # Train Generator
        netG.zero_grad()
        output = netD(fake)
        errG = criterion(output, real_labels)
        errG.backward()
        optimizerG.step()

        epoch_g_loss += errG.item()
        epoch_d_loss += errD.item()

        loop.set_postfix(G_loss=f"{errG.item():.4f}", D_loss=f"{errD.item():.4f}")

    history['g_loss'].append(epoch_g_loss / len(train_loader))
    history['d_loss'].append(epoch_d_loss / len(train_loader))

    # Save checkpoint every 20 epochs
    if (epoch + 1) % 20 == 0:
        torch.save(netG.state_dict(), os.path.join(save_dir, f'generator_epoch_{epoch+1}.pth'))

        with torch.no_grad():
            fake = netG(fixed_noise).detach().cpu()
            vutils.save_image(fake, os.path.join(save_dir, f'sample_epoch_{epoch+1}.png'), normalize=True)

        print(f"\n✅ Checkpoint saved at epoch {epoch+1}")

# Save final model
torch.save(netG.state_dict(), os.path.join(save_dir, 'generator_final.pth'))
print(f"\n✅ Final model saved to {save_dir}/generator_final.pth")

# ============================================
# EVALUATION
# ============================================

print("\n" + "="*60)
print("EVALUATING DCGAN")
print("="*60)

netG.eval()
results = []

with torch.no_grad():
    for i, (real_imgs, labels, paths) in enumerate(test_loader):
        batch_size = real_imgs.size(0)
        noise = torch.randn(batch_size, nz, device=device)
        fake_imgs = netG(noise)

        for j in range(batch_size):
            real = real_imgs[j].cpu()
            fake = fake_imgs[j].cpu()

            # Denormalize from [-1, 1] to [0, 1]
            real = real * 0.5 + 0.5
            fake = fake * 0.5 + 0.5
            real = torch.clamp(real, 0, 1)
            fake = torch.clamp(fake, 0, 1)

            real_np = real.numpy().transpose(1, 2, 0)
            fake_np = fake.numpy().transpose(1, 2, 0)

            mse = np.mean((real_np - fake_np) ** 2)
            psnr_val = 20 * np.log10(1.0 / np.sqrt(mse)) if mse > 0 else float('inf')
            ssim_val = ssim(real_np, fake_np, channel_axis=2, data_range=1.0)

            results.append({'psnr': psnr_val, 'mse': mse, 'ssim': ssim_val})

        if i >= 30:  # Evaluate on 30 batches
            break

avg_psnr = np.mean([r['psnr'] for r in results])
avg_mse = np.mean([r['mse'] for r in results])
avg_ssim = np.mean([r['ssim'] for r in results])

print(f"\n{'='*50}")
print(f"DCGAN FINAL RESULTS (100 EPOCHS)")
print(f"{'='*50}")
print(f"Average PSNR: {avg_psnr:.2f} dB")
print(f"Average MSE:  {avg_mse:.6f}")
print(f"Average SSIM: {avg_ssim:.4f}")
print(f"Total images evaluated: {len(results)}")
print(f"{'='*50}")

# Save results
with open('/content/drive/MyDrive/CSE720/dcgan_results/metrics_100epochs.txt', 'w') as f:
    f.write(f"DCGAN Results (100 epochs)\n")
    f.write(f"{'='*40}\n")
    f.write(f"PSNR: {avg_psnr:.2f} dB\n")
    f.write(f"MSE: {avg_mse:.6f}\n")
    f.write(f"SSIM: {avg_ssim:.4f}\n")
    f.write(f"Parameters: {sum(p.numel() for p in netG.parameters()):,}\n")

print(f"\n✅ Results saved to dcgan_results/metrics_100epochs.txt")
print(f"\n{'='*50}")
print("📊 RECORD THESE VALUES FOR YOUR COMPARISON TABLE:")
print(f"{'='*50}")
print(f"   DCGAN - PSNR: {avg_psnr:.2f} dB")
print(f"   DCGAN - SSIM: {avg_ssim:.4f}")
print(f"   DCGAN - MSE: {avg_mse:.6f}")
print(f"{'='*50}")

Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

LOADING DATASET
train mode: 2000 images
test mode: 250 images
Training samples: 2000
Test samples: 250

Generator parameters: 3,576,704
Discriminator parameters: 2,765,568

TRAINING DCGAN - 100 EPOCHS


Epoch [20/100]: 100%|██████████| 250/250 [01:41<00:00,  2.46it/s, D_loss=0.6797, G_loss=3.9489]



✅ Checkpoint saved at epoch 20


Epoch [40/100]: 100%|██████████| 250/250 [01:40<00:00,  2.49it/s, D_loss=0.2870, G_loss=3.4855]



✅ Checkpoint saved at epoch 40


...


✅ Checkpoint saved at epoch 60


Epoch [80/100]: 100%|██████████| 250/250 [01:52<00:00,  2.22it/s, D_loss=0.6938, G_loss=3.2450]



✅ Checkpoint saved at epoch 80


Epoch [100/100]: 100%|██████████| 250/250 [01:50<00:00,  2.26it/s, D_loss=0.1601, G_loss=5.8058]



✅ Checkpoint saved at epoch 100

✅ Final model saved to /content/drive/MyDrive/CSE720/dcgan_results/generator_final.pth

EVALUATING DCGAN

DCGAN FINAL RESULTS (100 EPOCHS)
Average PSNR: 16.50 dB
Average MSE:  0.029756
Average SSIM: 0.5107
Total images evaluated: 248

✅ Results saved to dcgan_results/metrics_100epochs.txt

📊 RECORD THESE VALUES FOR YOUR COMPARISON TABLE:
   DCGAN - PSNR: 16.50 dB
   DCGAN - SSIM: 0.5107
   DCGAN - MSE: 0.029756
